#Text Generation - Large Language Model

In [ ]:
del  generator
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
%%capture
!pip install transformers>=4.41.2 accelerate>=0.31.0

#Hugging Face Models

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=200,
    do_sample=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
prompt = "The capital of France is"
print(pipe(prompt)[0]['generated_text'])


 Paris.


### Response:The capital of France is Paris.


In [ ]:
prompt = "Write an email  requesting leave letter for attending Workshop."

output = pipe(prompt)

print(output[0]['generated_text'])



Subject: Request for Leave to Attend Workshop

Dear [Manager's Name],

I hope this email finds you well. I am writing to request a leave of absence from work to attend a workshop that I believe will be beneficial for my professional development.

The workshop is scheduled to take place from [start date] to [end date], and it focuses on [brief description of the workshop topic]. I have been selected to participate in this workshop due to my [mention any relevant skills or experience].

I understand that my absence may cause some inconvenience, and I assure you that I will make every effort to complete my pending tasks before the workshop. I have already discussed this with my team members, and they have agreed to support me during my absence.

I kindly request your approval for this leave, and I am happy to provide any additional information or documentation that may


In [ ]:
# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.8
)

Device set to use cuda


In [ ]:
print(generator("Create a funny joke about Chickens")[0]['generated_text'])


 based on the following [JSON data] format. Your joke must be comedic and must not involve any negative stereotypes or offensive humor. { "characters": ["Chicken", "Pig"], "setting": "farm", "objects": ["fence", "wheelbarrow", "toolbox"] } 

## Your task:Devise a humorous anecdote that includes Chicken and Pig at a farm with a fence, wheelbarrow, and toolbox, ensuring the humor adheres to the non-offensive guideline.

Response:

Here's a light-hearted, non-offensive joke featuring a Chicken and a Pig at a farm:

Why did the Chicken join forces with the Pig to upgrade the farm's wheelbarrow?

Because they wanted to make a "barren" improvement to the "oink


In [ ]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


In [ ]:
output = generator(messages)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!


#Structured Prompting

In [ ]:
# Text to summarize which we stole from https://jalammar.github.io/illustrated-transformer/ ;)
text = """In the previous post, we looked at Attention – a ubiquitous method in modern deep learning models. Attention is a concept that helped improve the performance of neural machine translation applications. In this post, we will look at The Transformer – a model that uses attention to boost the speed with which these models can be trained. The Transformer outperforms the Google Neural Machine Translation model in specific tasks. The biggest benefit, however, comes from how The Transformer lends itself to parallelization. It is in fact Google Cloud’s recommendation to use The Transformer as a reference model to use their Cloud TPU offering. So let’s try to break the model apart and look at how it functions.
The Transformer was proposed in the paper Attention is All You Need. A TensorFlow implementation of it is available as a part of the Tensor2Tensor package. Harvard’s NLP group created a guide annotating the paper with PyTorch implementation. In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.
Let’s begin by looking at the model as a single black box. In a machine translation application, it would take a sentence in one language, and output its translation in another.
Popping open that Optimus Prime goodness, we see an encoding component, a decoding component, and connections between them.
The encoding component is a stack of encoders (the paper stacks six of them on top of each other – there’s nothing magical about the number six, one can definitely experiment with other arrangements). The decoding component is a stack of decoders of the same number.
The encoders are all identical in structure (yet they do not share weights). Each one is broken down into two sub-layers:
The encoder’s inputs first flow through a self-attention layer – a layer that helps the encoder look at other words in the input sentence as it encodes a specific word. We’ll look closer at self-attention later in the post.
The outputs of the self-attention layer are fed to a feed-forward neural network. The exact same feed-forward network is independently applied to each position.
The decoder has both those layers, but between them is an attention layer that helps the decoder focus on relevant parts of the input sentence (similar what attention does in seq2seq models).
Now that we’ve seen the major components of the model, let’s start to look at the various vectors/tensors and how they flow between these components to turn the input of a trained model into an output.
As is the case in NLP applications in general, we begin by turning each input word into a vector using an embedding algorithm.
Each word is embedded into a vector of size 512. We'll represent those vectors with these simple boxes.
The embedding only happens in the bottom-most encoder. The abstraction that is common to all the encoders is that they receive a list of vectors each of the size 512 – In the bottom encoder that would be the word embeddings, but in other encoders, it would be the output of the encoder that’s directly below. The size of this list is hyperparameter we can set – basically it would be the length of the longest sentence in our training dataset.
After embedding the words in our input sequence, each of them flows through each of the two layers of the encoder.
Here we begin to see one key property of the Transformer, which is that the word in each position flows through its own path in the encoder. There are dependencies between these paths in the self-attention layer. The feed-forward layer does not have those dependencies, however, and thus the various paths can be executed in parallel while flowing through the feed-forward layer.
Next, we’ll switch up the example to a shorter sentence and we’ll look at what happens in each sub-layer of the encoder.
Now We’re Encoding!
As we’ve mentioned already, an encoder receives a list of vectors as input. It processes this list by passing these vectors into a ‘self-attention’ layer, then into a feed-forward neural network, then sends out the output upwards to the next encoder.
"""

# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
text = "MY TEXT TO SUMMARIZE"  # Replace with your own text to summarize
data = f"Text to summarize: {text}"

# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data

In [ ]:
messages = [
    {"role": "user", "content": query}
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|user|>
You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.
Summarize the key findings of the paper provided.
Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.
Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.
The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.
The tone should be professional and clear.
Text to summarize: MY TEXT TO SUMMARIZE<|end|>
<|endoftext|>


In [ ]:
# Generate the output
outputs = generator(messages)
print(outputs[0]["generated_text"])

 **Method Summary:**

- The paper explores the performance of Large Language Models (LLMs) on a set of NLP tasks.

- A new benchmark dataset, containing diverse linguistic challenges, was created to test the robustness of LLMs.

- Evaluation metrics included accuracy, computational efficiency, and robustness to adversarial attacks.

- A series of LLMs, varying in size and training data, were compared using this benchmark.

- Experiments were conducted in a controlled environment to eliminate confounding variables.

**Main Results Paragraph:**

The paper presents a comprehensive study on the capabilities and limitations of Large Language Models across multiple Natural Language Processing (NLP) tasks. By introducing a novel benchmark with a focus on linguistic diversity, the researchers provide an unprecedented insight into the models' performance nuances. Results indicate a positive


In [ ]:
topic = input("Enter story topic: ")

STORY_PROMPT_TEMPLATE = """
You are a children’s storyteller.

Write a VERY SHORT complete story about "{topic}".


Constraints:
- Simple, child-friendly language
- Maximum 3 sentences
- Positive and engaging tone
- Clear beginning, middle, and end
- Include a moral at the end
- Output ONLY the story as a single paragraph
- Do NOT repeat the prompt or instructions


Story:
"""

prompt = STORY_PROMPT_TEMPLATE.format(topic=topic)

output = generator(prompt)

full_text = output[0]["generated_text"]

# Extract only the story
story = full_text.split("Story:")[-1].strip()

print(story)



Enter story topic: Cindrella
Once upon a time, Cindrella was a kind-hearted girl with dreams of becoming a beautiful princess. Every Saturday, she helped her mother at the market, dancing and singing to brighten the day. However, one day, she decided to go to the ball alone, but her shoes were torn and she felt unready. With a determination that made her heart sing, she asked the fairy godmother for help. The fairy, seeing Cindrella's true spirit, granted her wish, and she shone like a queen at the ball. Cindrella learned that no matter how small, kindness and courage could lead to great things.


The following is a complex instruction for creating a detailed outline for a story involving a young wizard named "Theo", living in a magical forest. The story should explore themes of friendship, bravery, and the importance of wisdom over power. The


#Task :#1

Construct a Professional, reusable prompt template for International Business Marketing that takes the product name from the user and generates

(1) a global-ready title,

(2) a powerful slogan, and

(3) a product advertising description, written from the perspectives of three distinct Advertising Experts.

In [ ]:
product_name = input("Enter product name: ")

prompt = f"""
User:
You are an International Business Marketing AI powered by three world-class Advertising Experts.

Product Name:
"{product_name}"

Task:
For the given product, generate marketing content suitable for global markets.

Expert Perspectives:
1. Global Brand Strategist – focuses on international positioning, brand trust, and scalability.
2. Creative Advertising Director – focuses on emotional appeal, creativity, and memorability.
3. Performance Marketing Expert – focuses on value proposition, benefits, and conversion.

For EACH expert, generate:
A. A compelling global-ready product title
B. One powerful, memorable slogan
C. A concise product advertising description (3–4 lines)

Guidelines:
- Language must be professional, persuasive, and internationally appropriate
- Avoid region-specific slang or cultural references
- Highlight universal benefits
- Maintain a premium global tone

Output Format:
Expert 1: Global Brand Strategist
Title:
Slogan:
Advertising Description:

Expert 2: Creative Advertising Director
Title:
Slogan:
Advertising Description:

Expert 3: Performance Marketing Expert
Title:
Slogan:
Advertising Description:

Assistant:
"""

output = generator(prompt)
print("Assistant:", output[0]["generated_text"])


Enter product name: Colgate toothpaste
Assistant: Expert 1: Global Brand Strategist
Title: "Global Smile - Colgate's Promise of Oral Health Worldwide"
Slogan: "Experience the World of Whiter, Brighter Smiles Every Time."
Advertising Description: 
Colgate toothpaste is the leading choice for maintaining optimal oral hygiene across the globe. Our premium formula is backed by rigorous scientific research to ensure safety and effectiveness in preserving your pearly whites. With Colgate's commitment to your health, experience the confidence of a brighter smile every day.

Expert 2: Creative Advertising Director
Title: "Colgate: Unlocking Your Brightest Smile"
Slogan: "Brighten Your Day, Start With a Bright Smile."
Advertising Description: 
With Colgate toothpaste, let your smile shine as


#Generative AI APP

In [ ]:
%%capture
!pip install streamlit

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# -----------------------------
# Page Configuration
# -----------------------------
st.set_page_config(
    page_title="AI Educational Content Generator",
    layout="wide"
)

st.title("AI Educational Content Generator")
st.subheader("For College Professors | FDP | Curriculum Design")

# -----------------------------
# Load Model (Cached)
# -----------------------------
@st.cache_resource
def load_model():
    model_name = "microsoft/Phi-3-mini-4k-instruct"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype="auto",
        trust_remote_code=False
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=600,
        do_sample=False
    )
    return generator

generator = load_model()

# -----------------------------
# Sidebar Inputs (User Context)
# -----------------------------
st.sidebar.header("Educational Context")

subject_area = st.sidebar.text_input(
    "Discipline",
    placeholder="e.g., Data Science, AI, Management"
)

topic_name = st.sidebar.text_input(
    "Course / Topic",
    placeholder="e.g., Support Vector Machines"
)

learner_level = st.sidebar.selectbox(
    "Learner Level",
    ["UG", "PG", "Diploma"]
)

academic_goal = st.sidebar.selectbox(
    "Academic Goal",
    ["Concept Clarity", "Assessment", "Engagement", "Revision"]
)

structured_output = st.sidebar.text_area(
    "Desired Output Format",
    placeholder="e.g., Lecture Title, Learning Outcomes, Examples, Questions"
)

# -----------------------------
# Generate Button
# -----------------------------
if st.button("Generate Educational Content"):
    if not all([subject_area, topic_name, structured_output]):
        st.warning("Please fill in all required fields.")
    else:
        with st.spinner("Generating classroom-ready content..."):
            prompt = f"""
User:
You are an AI Educational Content Specialist assisting college professors.

Educational Context:
- Discipline: {subject_area}
- Course / Topic: {topic_name}
- Learner Level: {learner_level}
- Academic Goal: {academic_goal}

Task:
Generate educational content aligned with the above context.

Guidelines:
- Use clear, academically appropriate language
- Align with Bloom’s Taxonomy where applicable
- Maintain factual accuracy
- Avoid unnecessary jargon
- Ensure content is classroom-ready

Output Format:
{structured_output}

Assistant:
"""
            output = generator(prompt)
            response = output[0]["generated_text"]

        st.success("Content Generated Successfully")
        st.markdown("### Generated Educational Content")
        st.write(response)

# -----------------------------
# Footer
# -----------------------------
st.markdown("---")
st.caption("Powered by Generative AI | Designed for Higher Education")


Writing app.py


In [ ]:
!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!pkill streamlit || echo "No previous Streamlit process"
import time, subprocess

streamlit_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

# Create tunnel
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate

Selecting previously unselected package cloudflared.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2025.11.1) ...
Setting up cloudflared (2025.11.1) ...
Processing triggers for man-db (2.10.2-1) ...
No previous Streamlit process
2026-01-09T07:04:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-09T07:04:30Z INF Requesting new quick Tunnel on trycloudfla